In [8]:
import os

# J'ai ajouté cette cellule après avoir eu des PermissionError sur "../data" :
# VSCode connecté à un serveur Jupyter distant démarre parfois le kernel avec
# le mauvais dossier de travail (la racine du conteneur au lieu de notebooks/).
# Cette cellule force le bon cwd avant tout, comme dans les Sessions 3 et 4.
if os.path.basename(os.getcwd()) != "notebooks":
    os.chdir("/home/jovyan/notebooks")

print("cwd:", os.getcwd())

cwd: /home/jovyan/notebooks


# ClimaCity Paris -- Jour 1
## Exploration batch : l'API RDD et l'API DataFrame

**Module** : Traitement de donnees massives avec Apache Spark et PySpark  
**Duree** : 1 journee (6 heures effectives)  
**Prerequis** : Python intermediaire, notions de Pandas et NumPy

---

Ce notebook couvre l'integralite du Jour 1 du projet ClimaCity Paris.  
Il se divise en deux grandes parties :

- **Partie 1 -- Matin (3 h)** : l'API bas niveau RDD, la semantique de l'evaluation paresseuse, les transformations et actions fondamentales, et une premiere lecture du Spark UI.
- **Partie 2 -- Apres-midi (3 h)** : l'API haut niveau DataFrame, le nettoyage des donnees, la jointure avec les observations meteorologiques, la persistance en memoire et l'ecriture en Parquet partitionne.

> **Convention dans ce notebook**  
> Les cellules marquees `# [EXERCICE]` contiennent une consigne. Vous devez completer le code avant de passer a la cellule suivante.  
> Les cellules marquees `# [CORRECTION]` proposent une solution possible -- ne les regardez qu'apres avoir tente.


---
> **Transparence sur l'usage d'un assistant IA**
>
> Ce notebook a été relu et corrigé avec l'aide de l'assistant IA
> **Claude / Claude Code** (Anthropic) -- essentiellement des corrections
> ponctuelles (téléchargements rendus idempotents, variable manquante,
> problème d'environnement Docker), signalées individuellement par des
> commentaires à la première personne. Voir la section *« Transparence sur
> l'utilisation d'un assistant IA »* du `README.md` du projet pour le détail.
---


---
## Section 0 -- Configuration de l'environnement

Toutes les constantes du projet sont declarees ici. Adaptez les chemins si necessaire,
puis executez cette cellule avant toute autre.


In [9]:
import os
import time
from pathlib import Path

# ── Chemins de donnees ──────────────────────────────────────────────────────
DATA_DIR       = Path("../data")                          # racine des donnees
VELIB_RAW_DIR  = DATA_DIR / "velib" / "raw"              # CSV bruts (RDD)
VELIB_PARQ_DIR = DATA_DIR / "velib" / "parquet"          # Parquet partitionne
STATIONS_CSV   = DATA_DIR / "velib" / "stations_info.csv"
METEO_CSV      = DATA_DIR / "meteo" / "paris_montsouris_horaire.csv"
OUTPUT_DIR     = DATA_DIR / "output"                     # ecriture des resultats

# ── Parametres Spark ────────────────────────────────────────────────────────
APP_NAME       = "ClimaCity-Paris"
MASTER         = "local[*]"                              # toutes les CPU locales
SHUFFLE_PARTS  = 8                                       # adapte aux volumes du cours
SEED           = 42

# Verification des chemins
for p in [VELIB_RAW_DIR, VELIB_PARQ_DIR, STATIONS_CSV, METEO_CSV]:
    status = "OK" if p.exists() else "MANQUANT"
    print(f"[{status}] {p}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"[OK] {OUTPUT_DIR} (cree si absent)")


[OK] ../data/velib/raw
[MANQUANT] ../data/velib/parquet
[OK] ../data/velib/stations_info.csv
[MANQUANT] ../data/meteo/paris_montsouris_horaire.csv
[OK] ../data/output (cree si absent)


---
## Section 0.1 -- Collecte des données Vélib'

Deux sources complémentaires sont nécessaires :

- **Les informations statiques des stations** (nom, coordonnées, capacité) : publiées
  en temps réel par l'API GBFS de Vélib' Métropole, sans authentification.
- **L'historique de disponibilité** (snapshots toutes les 15 min depuis 2019) :
  disponible sur le dépôt GitHub `lovasoa/historique-velib-opendata`.

Exécutez les cellules ci-dessous **une seule fois** avant le démarrage de la session
Spark. Les fichiers sont écrits dans `DATA_DIR` et relus ensuite par PySpark.

In [10]:
import requests
import json
import pandas as pd

# J'ai ajouté cette vérification : si stations_info.csv existe déjà,
# pas besoin de rappeler l'API GBFS à chaque exécution du notebook.
if STATIONS_CSV.exists():
    df_stations = pd.read_csv(STATIONS_CSV, sep=";")
    print(f"[OK] {STATIONS_CSV} existe déjà ({len(df_stations)} stations) -- téléchargement ignoré.")
else:
    # ── Endpoint GBFS Vélib' Métropole (sans clé, sans inscription) ──────────────
    GBFS_STATION_INFO = (
        "https://velib-metropole-opendata.smovengo.cloud"
        "/opendata/Velib_Metropole/station_information.json"
    )

    print("Téléchargement des informations de stations...")
    resp = requests.get(GBFS_STATION_INFO, timeout=15)
    resp.raise_for_status()

    stations = resp.json()["data"]["stations"]
    print(f"  {len(stations)} stations reçues")

    df_stations = pd.DataFrame(stations)[[
        "station_id", "name", "lat", "lon", "capacity", "stationCode"
    ]]

    # Normalisation : station_id en entier (certaines valeurs sont des strings)
    df_stations["station_id"] = df_stations["station_id"].astype(int)

    # Extraction de l'arrondissement depuis le nom de la station
    # Convention Vélib' : le code commune figure dans station_id (75xxx = Paris)
    df_stations["code_arr"] = df_stations["station_id"] // 1000

    STATIONS_CSV.parent.mkdir(parents=True, exist_ok=True)
    df_stations.to_csv(STATIONS_CSV, index=False, sep=";")
    print(f"  Sauvegardé : {STATIONS_CSV}")

print(f"  Capacité totale du réseau : {df_stations['capacity'].sum():,} bornettes")
df_stations.head(5)

[OK] ../data/velib/stations_info.csv existe déjà (1519 stations) -- téléchargement ignoré.
  Capacité totale du réseau : 49,308 bornettes


,station_id,name,lat,lon,capacity,stationCode,code_arr
0,213688169,Benjamin Godard - Victor Hugo,48.865983,2.275725,35,16107,213688
1,19179944124,Hôpital Mondor,48.798922,2.453745,28,40001,19179944
2,18795078746,Charcot - Benfleet,48.878370,2.440524,28,32304,18795078
3,36255,Toudouze - Clauzel,48.879296,2.337360,21,9020,36
4,251039991,Cassini - Denfert-Rochereau,48.837526,2.336035,25,14111,251039


In [11]:
import urllib.request
import zipfile

URL = "https://github.com/lovasoa/historique-velib-opendata/releases/download/new/stations.zip"
DEST_ZIP = VELIB_RAW_DIR.parent / "stations.zip"
CSV_CIBLE = VELIB_RAW_DIR / "historique_stations.csv"

VELIB_RAW_DIR.mkdir(parents=True, exist_ok=True)

# J'ai ajouté cette vérification après avoir re-téléchargé le zip (~236 Mo)
# plusieurs fois par erreur en relançant le notebook depuis le début.
if CSV_CIBLE.exists():
    taille_mo = CSV_CIBLE.stat().st_size / 1_048_576
    print(f"[OK] {CSV_CIBLE} existe déjà ({taille_mo:.0f} Mo) -- téléchargement ignoré.")
else:
    print("Téléchargement de stations.zip (~236 Mo)...")
    urllib.request.urlretrieve(URL, DEST_ZIP)
    print("Téléchargement terminé.")

    print("Extraction...")
    with zipfile.ZipFile(DEST_ZIP, 'r') as z:
        print(f"  {len(z.namelist())} fichiers dans l'archive")
        print(f"  Premiers fichiers : {z.namelist()[:5]}")
        z.extractall(VELIB_RAW_DIR)

    print("Fait !")

[OK] ../data/velib/raw/historique_stations.csv existe déjà (780 Mo) -- téléchargement ignoré.


In [12]:
import pandas as pd

apercu = pd.read_csv(VELIB_RAW_DIR / "historique_stations.csv", nrows=5)
print(apercu.dtypes)
print("---")
print(apercu)

2020-11-26T12:59Z                object
35                                int64
4                                 int64
5                                 int64
Benjamin Godard - Victor Hugo    object
48.86598,2.27572                 object
True                               bool
dtype: object
---
   2020-11-26T12:59Z  35   4   5        Benjamin Godard - Victor Hugo  \
0  2020-11-26T12:59Z  55  23   4   André Mazet - Saint-André des Arts   
1  2020-11-26T12:59Z  20   0   0  Charonne - Robert et Sonia Delauney   
2  2020-11-26T12:59Z  21   0   1                   Toudouze - Clauzel   
3  2020-11-26T12:59Z  30   3   1                      Mairie du 12ème   
4  2020-11-26T12:59Z  46  18  10                Harpe - Saint-Germain   

   48.86598,2.27572  True  
0  48.85376,2.33910  True  
1  48.85591,2.39257  True  
2  48.87930,2.33736  True  
3  48.84086,2.38755  True  
4  48.85152,2.34367  True  


In [13]:
apercu = pd.read_csv(
    VELIB_RAW_DIR / "historique_stations.csv",
    header=None,
    names=["horodatage", "capacite", "velos_dispo", "velos_elec", "nom_station", "coordonnees", "is_installed"],
    nrows=5
)
print(apercu)
print("---")
import subprocess
result = subprocess.run(["wc", "-l", str(VELIB_RAW_DIR / "historique_stations.csv")], capture_output=True, text=True)
print(f"Nombre de lignes total : {result.stdout.strip()}")

          horodatage  capacite  velos_dispo  velos_elec  \
0  2020-11-26T12:59Z        35            4           5   
1  2020-11-26T12:59Z        55           23           4   
2  2020-11-26T12:59Z        20            0           0   
3  2020-11-26T12:59Z        21            0           1   
4  2020-11-26T12:59Z        30            3           1   

                           nom_station       coordonnees  is_installed  
0        Benjamin Godard - Victor Hugo  48.86598,2.27572          True  
1   André Mazet - Saint-André des Arts  48.85376,2.33910          True  
2  Charonne - Robert et Sonia Delauney  48.85591,2.39257          True  
3                   Toudouze - Clauzel  48.87930,2.33736          True  
4                      Mairie du 12ème  48.84086,2.38755          True  
---
Nombre de lignes total : 10986730 ../data/velib/raw/historique_stations.csv


# **Nota bene**
- Le téléchargement complet (2 ans) représente environ 800 MB à 1,5 GB selon les périodes couvertes par les releases ; prévoir 15 à 30 minutes selon la connexion.
- L'API GitHub Releases est non authentifiée mais limitée à 60 requêtes/heure par IP. Un seul appel à REPO_API suffit pour lister tout l'historique.


---
# PARTIE 1 -- L'API RDD (matin)

## 1.1 La SparkSession et le SparkContext

### Pourquoi Spark ?

Pandas est un outil remarquable pour les donnees qui tiennent en memoire sur une seule
machine. Mais il atteint ses limites lorsque le volume depasse quelques gigaoctets, ou
lorsque le traitement doit s'executer en parallele sur plusieurs coeurs ou plusieurs noeuds.

Apache Spark repond a ce besoin en proposant un modele de calcul distribue et tolerant aux
pannes, dans lequel les donnees sont representees comme des collections immuables et
partitionnees, les **RDD** (*Resilient Distributed Datasets*).

Meme en mode local -- sur une seule machine, comme c'est le cas dans ce cours -- Spark
parallelise les traitements sur tous les coeurs disponibles et permet de travailler sur
des volumes qui depassent la memoire vive grace a la gestion automatique du debordement
sur disque.

### Le point d'entree : `SparkSession`

Depuis Spark 2.0, `SparkSession` est le point d'entree unifie. Il encapsule le
`SparkContext` (execution RDD), le `SQLContext` (SQL et DataFrame) et le
`HiveContext`. On n'en cree qu'une par application.


In [ ]:
from pyspark.sql import SparkSession
from pyspark import StorageLevel

spark = (
    SparkSession.builder
    .appName("ClimaCity-Paris")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

# Le SparkContext est accessible depuis la session
sc = spark.sparkContext
sc.setLogLevel("WARN")

print(f"Spark version    : {spark.version}")
print(f"Python version   : {sc.pythonVer}")
print(f"Master           : {sc.master}")
print(f"Coeurs detectes  : {sc._jsc.sc().defaultParallelism()}")
print(f"\nSpark UI disponible sur : http://localhost:4040")

Spark version    : 3.5.3
Python version   : 3.11
Master           : local[*]
Coeurs detectes  : 4

Spark UI disponible sur : http://localhost:4040


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 38918)
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/opt/conda/lib/python3.11/socketserver.py", line 755, in __init__
    self.handle()
  File "/usr/local/spark/python/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/usr/local/spark/python/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
                           ^^^^^^
  File "/usr/local/spark/python/pyspark/accumulators.py", line 271, in accum_updates
    num_updates =

> **Spark UI** : ouvrez l'adresse affichee ci-dessus dans votre navigateur. Vous trouverez
> l'onglet **Jobs** (vide pour l'instant), **Stages**, **Storage** et **Environment**.
> Gardez-le ouvert en permanence pendant cette journee.

### Structure d'un cluster Spark (meme en local)

```
SparkSession (driver)
  |
  +-- SparkContext
        |
        +-- Executor 1 (Thread pool)
        |     +-- Task 1-1
        |     +-- Task 1-2
        +-- Executor 2 (Thread pool)
              +-- Task 2-1
              +-- Task 2-2
```

En mode `local[*]`, le driver et les executeurs partagent le meme processus JVM.
Chaque coeur logique correspond a un *slot* d'execution de taches.


---
## 1.2 Chargement des donnees brutes avec `sc.textFile()`

Le jeu de donnees brut se presente sous forme de fichiers CSV compresses (`.csv.gz`),
un par quinzaine de jours environ. Chaque ligne decrit l'etat d'une station a un instant
donne. Voici le format :

```
station_id;nom_station;code_arrondissement;capacite;velos_meca;velos_elec;bornettes_libres;horodatage
1001;Bir-Hakeim - Grenelle;75015;25;12;3;10;2022-01-01T00:14:52+00:00
1002;Javel - Andre Citroen;75015;30;0;0;30;2022-01-01T00:14:55+00:00
...
```

L'API `sc.textFile()` retourne un **RDD de chaines de caracteres** -- une ligne du fichier
par element. Aucune interpretation du contenu n'est effectuee a ce stade.


In [15]:
raw_rdd = sc.textFile(str(VELIB_RAW_DIR / "historique_stations.csv"))

n_lignes = raw_rdd.count()
print(f"Nombre de lignes : {n_lignes:,}")

for ligne in raw_rdd.take(5):
    print(ligne)

Nombre de lignes : 10,986,730
2020-11-26T12:59Z,35,4,5,Benjamin Godard - Victor Hugo,"48.86598,2.27572",True
2020-11-26T12:59Z,55,23,4,André Mazet - Saint-André des Arts,"48.85376,2.33910",True
2020-11-26T12:59Z,20,0,0,Charonne - Robert et Sonia Delauney,"48.85591,2.39257",True
2020-11-26T12:59Z,21,0,1,Toudouze - Clauzel,"48.87930,2.33736",True
2020-11-26T12:59Z,30,3,1,Mairie du 12ème,"48.84086,2.38755",True


### Observations

- `sc.textFile()` est **instantane** : il ne lit pas encore les fichiers. Il cree
  seulement un *plan de calcul* (un RDD logique).
- `count()` declenche la lecture et le comptage -- c'est la premiere **action**.
- Les fichiers sont decoupes en **partitions** (une par bloc HDFS ou par fichier local).
  Le nombre de partitions determine le parallelisme maximal.

Verifions le nombre de partitions :


In [16]:
print(f"Nombre de partitions : {raw_rdd.getNumPartitions()}")

# En mode local, Spark cree typiquement une partition par coeur,
# ou une par fichier si le nombre de fichiers est superieur au nombre de coeurs.
# On peut forcer un repartitionnement :
raw_rdd_8p = raw_rdd.repartition(8)
print(f"Apres repartition(8) : {raw_rdd_8p.getNumPartitions()} partitions")
# Note : repartition() est une transformation -- elle ne s'execute pas encore.


Nombre de partitions : 25
Apres repartition(8) : 8 partitions


---
## 1.3 Transformations elementaires : `map`, `filter`

Un RDD est une collection **immuable** et **paresseuse** : les transformations ne
s'executent qu'au moment ou une action les requiert.

### Separation de l'en-tete

La premiere ligne de chaque fichier est un en-tete. Il faut l'eliminer avant de parser
les donnees.


In [17]:
entete = raw_rdd.first()
print(f"Première ligne : {entete}")

donnees_rdd = raw_rdd.filter(lambda ligne: ligne != entete)
print(f"Nombre de lignes sans la première : {donnees_rdd.count():,}")

Première ligne : 2020-11-26T12:59Z,35,4,5,Benjamin Godard - Victor Hugo,"48.86598,2.27572",True
Nombre de lignes sans la première : 10,986,729


### Parsing des lignes avec `map()`

`map(f)` applique la fonction `f` a chaque element du RDD et retourne un nouveau RDD
de meme longueur contenant les resultats. C'est une transformation *un-pour-un*.


In [18]:
from datetime import datetime, timezone
import csv
import io

COLONNES = [
    "horodatage", "capacite", "velos_meca", "velos_elec",
    "nom_station", "coordonnees", "is_installed"
]

def parse_ligne(line: str) -> dict | None:
    """Parse une ligne CSV du fichier Vélib' brut.

    Args:
        line: Chaîne brute séparée par des virgules.

    Returns:
        Dictionnaire avec les champs typés, ou None si la ligne est malformée.
    """
    try:
        champs = next(csv.reader(io.StringIO(line)))
        if len(champs) != 7:
            return None
        lat, lon = champs[5].split(",")
        return {
            "horodatage":   champs[0],
            "capacite":     int(champs[1]),
            "velos_meca":   int(champs[2]),
            "velos_elec":   int(champs[3]),
            "nom_station":  champs[4].strip(),
            "lat":          float(lat),
            "lon":          float(lon),
            "is_installed": champs[6].strip() == "True",
        }
    except Exception:
        return None

parsed_rdd = donnees_rdd.map(parse_ligne).filter(lambda x: x is not None)
print(f"Lignes parsées valides : {parsed_rdd.count():,}")
for r in parsed_rdd.take(2):
    print(r)

Lignes parsées valides : 10,986,729
{'horodatage': '2020-11-26T12:59Z', 'capacite': 55, 'velos_meca': 23, 'velos_elec': 4, 'nom_station': 'André Mazet - Saint-André des Arts', 'lat': 48.85376, 'lon': 2.3391, 'is_installed': True}
{'horodatage': '2020-11-26T12:59Z', 'capacite': 20, 'velos_meca': 0, 'velos_elec': 0, 'nom_station': 'Charonne - Robert et Sonia Delauney', 'lat': 48.85591, 'lon': 2.39257, 'is_installed': True}


> **Examen du Spark UI** : allez dans l'onglet **Jobs**. Vous devriez voir deux jobs
> termines. Cliquez sur le dernier (le `count()`). Dans la vue **Stages**, observez :
>
> - Le nombre de taches (*tasks*) -- une par partition.
> - Le temps passe dans chaque tache.
> - La colonne **Input** : la quantite de donnees lues par chaque tache.
>
> Notez que les deux `count()` que nous avons declenches ont relu les fichiers
> depuis le disque a chaque fois. C'est le comportement par defaut -- nous verrons
> comment l'eviter avec `.cache()`.


---
## 1.4 Evaluation paresseuse : transformations vs actions

C'est le concept le plus important de Spark. Toutes les **transformations** sont
**paresseuses** : elles construisent un plan de calcul (un DAG), mais ne font rien.
Seules les **actions** declenchent l'execution.

| Transformations (paresseuses) | Actions (declenchent le calcul) |
|-------------------------------|----------------------------------|
| `map()`, `filter()`, `flatMap()` | `count()`, `collect()`, `take()` |
| `groupByKey()`, `reduceByKey()` | `first()`, `top()`, `reduce()` |
| `join()`, `union()`, `distinct()` | `saveAsTextFile()`, `foreach()` |
| `repartition()`, `coalesce()` | `countByValue()`, `countByKey()` |

Illustrons ce comportement :


In [19]:
import time

t0 = time.perf_counter()

# Étape 1 : garder uniquement les stations installées et non vides
step1 = parsed_rdd.filter(lambda r: r["is_installed"] and r["capacite"] > 0)

# Étape 2 : ajouter le nombre total de vélos et le taux d'occupation
step2 = step1.map(lambda r: {
    **r,
    "velos_total": r["velos_meca"] + r["velos_elec"],
    "taux_occupation": round(
        (r["velos_meca"] + r["velos_elec"]) / r["capacite"], 4
    )
})

# Étape 3 : garder les snapshots avec moins de 10% d'occupation (stations quasi-vides)
step3 = step2.filter(lambda r: r.get("taux_occupation", 0) < 0.1)

t1 = time.perf_counter()
print(f"Construction du plan (3 transformations) : {(t1-t0)*1000:.1f} ms")

t2 = time.perf_counter()
n = step3.count()
t3 = time.perf_counter()

print(f"Execution de count() (lecture + calcul) : {(t3-t2):.2f} s")
print(f"Lignes avec taux < 10% : {n:,}")

Construction du plan (3 transformations) : 0.1 ms
Execution de count() (lecture + calcul) : 12.12 s
Lignes avec taux < 10% : 1,722,926


### Le DAG (Directed Acyclic Graph)

Spark traduit votre chaine de transformations en un graphe oriente acyclique.
Ce graphe est optimise avant l'execution par le **Catalyst optimizer** (pour les
DataFrames) ou execute tel quel (pour les RDD).

```
sc.textFile()  -->  filter(header)  -->  map(parse)  -->  filter(None)
    -->  filter(capacite > 0)  -->  map(taux)  -->  filter(valide)
    -->  count()  [ACTION]
```

> **Spark UI -> Jobs -> dernier job -> DAG Visualization** : vous voyez exactement
> ce graphe. Les boites bleues sont les **stages** (separees par des shuffles).
> Une fleche grise entre deux stages indique un shuffle (transfert de donnees
> entre executeurs).


In [20]:
# On va reutiliser step3 plusieurs fois -> bonne occasion de le mettre en cache
# .cache() est equivalent a .persist(StorageLevel.MEMORY_ONLY)
step3.cache()

# Premier acces : lecture disque + mise en cache
t0 = time.perf_counter()
n1 = step3.count()
t1 = time.perf_counter()
print(f"Premier count() (lecture disque) : {t1-t0:.2f} s -- {n1:,} lignes")

# Deuxieme acces : depuis le cache en memoire
t2 = time.perf_counter()
n2 = step3.count()
t3 = time.perf_counter()
print(f"Deuxieme count() (depuis cache)  : {t3-t2:.2f} s -- {n2:,} lignes")

print(f"\nGain du cache : x{((t1-t0)/(t3-t2)):.1f}")
print("\nAllez dans Spark UI -> Storage pour voir le RDD en cache.")


Premier count() (lecture disque) : 13.70 s -- 1,722,926 lignes
Deuxieme count() (depuis cache)  : 0.76 s -- 1,722,926 lignes

Gain du cache : x18.0

Allez dans Spark UI -> Storage pour voir le RDD en cache.


---
## 1.5 Agregations avec paires cle-valeur : `reduceByKey`

L'une des operations les plus importantes de l'API RDD est `reduceByKey()`. Elle
regroupe les elements qui partagent la meme cle et combine leurs valeurs avec une
fonction associative.

### Objectif : top 10 des stations par nombre de snapshots

Nous voulons savoir quelles stations apparaissent le plus souvent dans notre historique
(indicateur indirect d'une bonne couverture de collecte).


In [21]:
paires_rdd = parsed_rdd.map(lambda r: (r["nom_station"], 1))

print("Exemples de paires :")
for p in paires_rdd.take(5):
    print(" ", p)

Exemples de paires :
  ('André Mazet - Saint-André des Arts', 1)
  ('Charonne - Robert et Sonia Delauney', 1)
  ('Toudouze - Clauzel', 1)
  ('Mairie du 12ème', 1)
  ('Harpe - Saint-Germain', 1)


In [22]:
comptage_rdd = paires_rdd.reduceByKey(lambda a, b: a + b)

top10_rdd = comptage_rdd.sortBy(lambda x: -x[1])

top10 = top10_rdd.take(10)

print("Top 10 des stations par nombre de snapshots :")
print(f"{'Station':<40} {'Snapshots':>12}")
print("-" * 54)
for station, count in top10:
    print(f"{station:<40} {count:>12,}")

Top 10 des stations par nombre de snapshots :
Station                                     Snapshots
------------------------------------------------------
Place Nelson Mandela                           15,732
Place de l'Eglise                              15,732
Place de l'Europe                              15,732
Université                                     15,732
Château - République                           15,732
Hôpital Européen Georges Pompidou               7,868
Gare du Nord - Hôpital Lariboisière             7,866
Panthéon - Valette                              7,866
Cimetière de Montmartre                         7,866
Grands Moulins de Pantin                        7,866


### `reduceByKey` vs `groupByKey` : une difference critique

`groupByKey()` regroupe *toutes* les valeurs en memoire avant de les reduire.
Sur de gros volumes, cela peut provoquer des `OutOfMemoryError`.
`reduceByKey()` effectue une **combinaison locale** avant le shuffle, ce qui reduit
drastiquement le volume de donnees transferees.

```
groupByKey()   : SHUFFLE toutes les valeurs -> reduire
reduceByKey()  : reduire localement -> SHUFFLE resultats partiels -> reduire
```

Regle pratique : **ne jamais utiliser `groupByKey()` quand `reduceByKey()` suffit**.


In [23]:
import time

t0 = time.perf_counter()
_ = paires_rdd.groupByKey().mapValues(sum).count()
t_group = time.perf_counter() - t0

t0 = time.perf_counter()
_ = paires_rdd.reduceByKey(lambda a, b: a + b).count()
t_reduce = time.perf_counter() - t0

print(f"groupByKey + mapValues(sum) : {t_group:.2f} s")
print(f"reduceByKey                 : {t_reduce:.2f} s")
print(f"Rapport                     : x{t_group/t_reduce:.1f}")

groupByKey + mapValues(sum) : 12.17 s
reduceByKey                 : 11.48 s
Rapport                     : x1.1


---
## 1.6 `flatMap` et `join` entre deux RDD

### `flatMap` : une entree, zero ou plusieurs sorties

Contrairement a `map` (toujours une sortie par entree), `flatMap` peut retourner
un nombre quelconque d'elements pour chaque element d'entree. C'est utile pour
"eclater" des structures imbriquees.


In [24]:
# Exemple pedagogique : a partir de chaque snapshot, on veut produire
# autant de paires (heure_de_la_journee, taux_occupation) que necessaire
# pour alimenter une distribution horaire.

def extraire_heure_et_taux(record: dict) -> list[tuple]:
    try:
        heure = int(record["horodatage"][11:13])
        return [(heure, record["taux_occupation"])]
    except Exception:
        return []

# Question : Quelle est la forme du résultat `heure_taux_rdd`
heure_taux_rdd = step3.flatMap(extraire_heure_et_taux)

"""
Exercice 13 :
-------------
# Calculer la moyenne du taux d'occupation par heure de la journee
# Astuce : on somme (taux, 1) puis on divise
"""
somme_count_rdd = heure_taux_rdd.map(lambda x: (x[0], (x[1], 1)))
agregat_rdd     = somme_count_rdd.reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1]))
moyenne_rdd     = agregat_rdd.map(lambda x: (x[0], round(x[1][0]/x[1][1], 4)))
profil_horaire  = moyenne_rdd.sortByKey().collect()

print("Taux d'occupation moyen par heure de la journee (toutes stations)")
print(f"{'Heure':<8} {'Taux moyen':>12}")
print("-" * 22)
for heure, taux in profil_horaire:
    barre = "#" * int(taux * 30)
    print(f"  {heure:02d}h    {taux:.4f}  {barre}")


Taux d'occupation moyen par heure de la journee (toutes stations)
Heure      Taux moyen
----------------------
  00h    0.0540  #
  01h    0.0544  #
  02h    0.0546  #
  03h    0.0553  #
  04h    0.0554  #
  05h    0.0558  #
  06h    0.0550  #
  07h    0.0518  #
  08h    0.0496  #
  09h    0.0494  #
  10h    0.0491  #
  11h    0.0486  #
  12h    0.0490  #
  13h    0.0489  #
  14h    0.0485  #
  15h    0.0492  #
  16h    0.0506  #
  17h    0.0517  #
  18h    0.0514  #
  19h    0.0515  #
  20h    0.0524  #
  21h    0.0532  #
  22h    0.0534  #
  23h    0.0538  #


### `join` entre deux RDD

On peut joindre deux RDD de paires `(cle, valeur)` comme on joinderait deux tables.


In [25]:
noms_rdd = parsed_rdd.map(lambda r: (r["nom_station"], r["nom_station"])).distinct()

velos_rdd = step2.map(lambda r: (r["nom_station"], r["velos_total"])) \
                 .reduceByKey(lambda a, b: a + b)

joint_rdd = noms_rdd.join(velos_rdd)

top5 = joint_rdd.sortBy(lambda x: -x[1][1]).take(5)

print("Top 5 des stations par volume cumulé de vélos disponibles :")
for nom, (_, total) in top5:
    print(f"  {nom:<45} {total:>10,}")

Top 5 des stations par volume cumulé de vélos disponibles :
  Emeriau - Beaugrenelle                           422,058
  Grenelle - Dr Finlay                             398,057
  Parc André Citroën                               362,519
  Bourdonnais - Tour Eiffel                        352,836
  Regnault - Patay                                 340,502


---
## 1.7 Comparaison Pandas vs Spark

Reprenons le calcul "top 10 des stations par nombre de snapshots" en Pandas et
comparons les temps et la demarche.


In [26]:
import pandas as pd

# --- Version Pandas ---
t0 = time.perf_counter()
df_pandas = pd.read_csv(
    VELIB_RAW_DIR / "historique_stations.csv",
    header=None,
    names=["horodatage", "capacite", "velos_meca", "velos_elec",
           "nom_station", "coordonnees", "is_installed"]
)
top10_pandas = df_pandas["nom_station"].value_counts().head(10)
t_pandas = time.perf_counter() - t0

# --- Version Spark (déjà calculée) ---
t0 = time.perf_counter()
top10_spark = top10_rdd.take(10)
t_spark = time.perf_counter() - t0

print(f"Pandas : {t_pandas:.2f} s  --  {len(df_pandas):,} lignes chargées")
print(f"Spark  : {t_spark:.2f} s  --  depuis le cache RDD")
print()
print(f"{'Station':<45} {'Pandas':>8}  {'Spark':>8}")
print("-" * 65)
for i in range(5):
    p_nom, p_cnt = top10_pandas.index[i], top10_pandas.iloc[i]
    s_nom, s_cnt = top10_spark[i]
    print(f"{p_nom:<45} {p_cnt:>8,}  {s_cnt:>8,}")

Pandas : 8.38 s  --  10,986,730 lignes chargées
Spark  : 0.18 s  --  depuis le cache RDD

Station                                         Pandas     Spark
-----------------------------------------------------------------
Place de l'Europe                               15,732    15,732
Université                                      15,732    15,732
Place de l'Eglise                               15,732    15,732
Château - République                            15,732    15,732
Benjamin Godard - Victor Hugo                    7,866    15,732


In [27]:
# --- Analyse de la memoire ---
mem_pandas_mb = df_pandas.memory_usage(deep=True).sum() / 1_048_576
print(f"\nMemoire occupee par le DataFrame Pandas  : {mem_pandas_mb:.1f} MB")
print(f"Memoire disponible pour le reste de l'OS : limitee")
print()
print("Conclusion :")
print("  - Sur ce volume, Pandas est souvent plus rapide (pas d'overhead JVM/reseau)")
print("  - Spark devient avantageux quand :")
print("    * Le volume depasse la RAM disponible")
print("    * Le calcul peut etre parallelise sur un cluster")
print("    * On combine batch + streaming + ML dans le meme pipeline")
print("    * On travaille en equipe sur un cluster partage")



Memoire occupee par le DataFrame Pandas  : 2733.3 MB
Memoire disponible pour le reste de l'OS : limitee

Conclusion :
  - Sur ce volume, Pandas est souvent plus rapide (pas d'overhead JVM/reseau)
  - Spark devient avantageux quand :
    * Le volume depasse la RAM disponible
    * Le calcul peut etre parallelise sur un cluster
    * On combine batch + streaming + ML dans le meme pipeline
    * On travaille en equipe sur un cluster partage
